In [ ]:
#Installera kraken
!apt-get -q install poppler-utils

## Install Python libraries used in the notebook
# pdf2image: convert PDF pages to images
!pip -q install pdf2image pymupdf pandas numpy matplotlib


Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
# Load PDF from Google Colab or use a local fallback
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf"
    print(f"Running locally, using: {pdf_path}")
    

Saving aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf to aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2 (1).pdf


In [12]:
##Extract GROUND TRUTH from PDF text layer
import fitz
import pandas as pd

doc = fitz.open(pdf_path)

gt_rows = []
for i in range(len(doc)):
    gt_rows.append({"page": i+1, "gt_text": doc[i].get_text("text").strip()})

gt_df = pd.DataFrame(gt_rows)
print("Pages found:", len(gt_df))
gt_df.to_csv("/content/groundtruth_pages_kraken.csv", index=False)



Pages found: 68


In [15]:
#Create groundtruth_ALL.md
import os

md_gt_path = "/content/groundtruth_ALL.md"

lines = ["# Ground Truth Text (Whole PDF)\n"]
for _, row in gt_df.iterrows():
    page = int(row["page"])
    text = (row["gt_text"] or "").strip()

    lines.append(f"## Page {page}\n")
    lines.append(text + "\n")

with open(md_gt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Created:", md_gt_path, "size:", os.path.getsize(md_gt_path))


Created: /content/groundtruth_ALL.md size: 211976


In [16]:
#Create groundtruth_ALL.txt
txt_gt_path = "/content/groundtruth_ALL.txt"

with open(txt_gt_path, "w", encoding="utf-8") as f:
    for _, row in gt_df.iterrows():
        f.write(f"=== PAGE {int(row['page'])} ===\n")
        f.write((row["gt_text"] or "").strip() + "\n\n")

print("Created:", txt_gt_path, "size:", os.path.getsize(txt_gt_path))


Created: /content/groundtruth_ALL.txt size: 212216


In [17]:
#Convert PDF → page images
from pdf2image import convert_from_path

pages = convert_from_path(pdf_path, dpi=300)

img_paths = []
for i, img in enumerate(pages, start=1):
    p = f"/content/kraken_page_{i:03d}.png"
    img.save(p, "PNG")
    img_paths.append(p)

print("Images created:", len(img_paths))
print("First:", img_paths[0])


Images created: 68
First: /content/kraken_page_001.png


In [ ]:
#Install Kraken in a separate micromamba environment
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba >/dev/null
!./bin/micromamba create -y -n kraken310 python=3.10 pip >/dev/null
!./bin/micromamba run -n kraken310 pip install -q "kraken==4.3.13" pandas numpy matplotlib >/dev/null
!MPLBACKEND=Agg ./bin/micromamba run -n kraken310 kraken --version



/root/.local/share/mamba/envs/kraken310/lib/python3.10/site-packages/kraken/kraken.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
kraken, version 4.3.13


In [ ]:
# Download the pretrained OCR model used by Kraken
!rm -f /content/catmus-print-fondue-small-2024-01-31.mlmodel
!wget -O /content/catmus-print-fondue-small-2024-01-31.mlmodel "https://zenodo.org/records/10602307/files/catmus-print-fondue-small-2024-01-31.mlmodel?download=1"
!ls -lh /content/catmus-print-fondue-small-2024-01-31.mlmodel
!file /content/catmus-print-fondue-small-2024-01-31.mlmodel


--2026-01-13 12:29:04--  https://zenodo.org/records/10602307/files/catmus-print-fondue-small-2024-01-31.mlmodel?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.52.235, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1705182 (1.6M) [application/octet-stream]
Saving to: ‘/content/catmus-print-fondue-small-2024-01-31.mlmodel’

/content/catmus-pri 100%[===================>]   1.63M   673KB/s    in 2.5s    

2026-01-13 12:29:08 (673 KB/s) - ‘/content/catmus-print-fondue-small-2024-01-31.mlmodel’ saved [1705182/1705182]

-rw-r--r-- 1 root root 1.7M Jan 13 12:29 /content/catmus-print-fondue-small-2024-01-31.mlmodel
/content/catmus-print-fondue-small-2024-01-31.mlmodel: PDP-11 pure executable not stripped - version 26


In [20]:
#for missing pages
import os, subprocess

kraken_model = "/content/catmus-print-fondue-small-2024-01-31.mlmodel"
out_dir = "/content/kraken_out"
os.makedirs(out_dir, exist_ok=True)

for i, img in enumerate(img_paths, start=1):
    out_txt = f"{out_dir}/page_{i:03d}.txt"

    # skip pages already finished
    if os.path.exists(out_txt) and os.path.getsize(out_txt) > 20:
        continue

    print(f"[Kraken resume] page {i}/{len(img_paths)}")
    cmd = f"MPLBACKEND=Agg ./bin/micromamba run -n kraken310 kraken -i {img} {out_txt} segment -bl ocr -m {kraken_model}"
    subprocess.run(["bash","-lc", cmd], check=True)

print("All missing pages processed.")


[Kraken resume] page 1/68
[Kraken resume] page 3/68
[Kraken resume] page 9/68
[Kraken resume] page 19/68
[Kraken resume] page 20/68
[Kraken resume] page 21/68
[Kraken resume] page 22/68
[Kraken resume] page 23/68
[Kraken resume] page 24/68
[Kraken resume] page 25/68
[Kraken resume] page 26/68
[Kraken resume] page 27/68
[Kraken resume] page 28/68
[Kraken resume] page 29/68
[Kraken resume] page 30/68
[Kraken resume] page 31/68
[Kraken resume] page 32/68
[Kraken resume] page 33/68
[Kraken resume] page 34/68
[Kraken resume] page 35/68
[Kraken resume] page 36/68
[Kraken resume] page 37/68
[Kraken resume] page 38/68
[Kraken resume] page 39/68
[Kraken resume] page 40/68
[Kraken resume] page 41/68
[Kraken resume] page 42/68
[Kraken resume] page 43/68
[Kraken resume] page 44/68
[Kraken resume] page 45/68
[Kraken resume] page 46/68
[Kraken resume] page 47/68
[Kraken resume] page 48/68
[Kraken resume] page 49/68
[Kraken resume] page 50/68
[Kraken resume] page 51/68
[Kraken resume] page 52/68
[Kra

In [ ]:
#Build predictions CSV
import pandas as pd

rows = []
for i in range(1, len(img_paths)+1):
    p = f"/content/kraken_out/page_{i:03d}.txt"
    with open(p, "r", encoding="utf-8", errors="ignore") as f:
        rows.append({"page": i, "pred_text": f.read().strip()})

pred_df = pd.DataFrame(rows)
pred_df.to_csv("/content/predictions_pages_kraken.csv", index=False)
print("Saved: /content/predictions_pages_kraken.csv")
pred_df.head()


Saved: /content/predictions_pages_kraken.csv


,page,pred_text
0,1,1a.lse
1,2,1501.1aee\nM1.1aMldeelatieeMEielalz Ior.185r7e...
2,3,
3,4,Vàr historia\nFrän melaniskt till robottelni...
4,5,SFlexQubes Arsredovisning 2022\n3procent.\nAre...


In [ ]:
# Create Kraken output markdown for whole PDF
out_path = "/content/output_kraken_ALL.md"

with open(out_path, "w", encoding="utf-8") as f:
    f.write("# Kraken OCR output (whole PDF)\n\n")
    for _, row in pred_df.iterrows():
        f.write(f"## Page {row['page']}\n\n")
        f.write(row['pred_text'] + "\n\n")

print("Saved:", out_path)


Saved: /content/output_kraken_ALL.md


In [ ]:
# Download outputs
from google.colab import files

files.download("/content/groundtruth_pages_kraken.csv")
files.download("/content/groundtruth_ALL.md")
files.download("/content/groundtruth_ALL.txt")
files.download("/content/predictions_pages_kraken.csv")
files.download("/content/output_kraken_ALL.md")
